In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
CREATE OR REPLACE TABLE workspace.gold.company_kpis AS

WITH pivoted AS (
  SELECT
    company,
    report_year,
    MAX(CASE WHEN kpi_name = 'Scope 1 GHG emissions'                  THEN value END) AS scope1_t,
    MAX(CASE WHEN kpi_name = 'Scope 2 GHG emissions (location-based)' THEN value END) AS scope2_location_t,
    MAX(CASE WHEN kpi_name = 'Scope 2 GHG emissions (market-based)'   THEN value END) AS scope2_market_t,
    MAX(CASE WHEN kpi_name = 'Scope 3 GHG emissions'                  THEN value END) AS scope3_t,
    MAX(CASE WHEN kpi_name = 'Total energy consumption'               THEN value END) AS energy_mwh,
    MAX(CASE WHEN kpi_name = 'Share of renewable energy consumption'  THEN value END) AS renewable_share_pct,
    MAX(CASE WHEN kpi_name = 'Employees (headcount)'                  THEN value END) AS employees,
    MAX(CASE WHEN kpi_name = 'Female employees (headcount)'           THEN value END) AS female_employees,
    MAX(CASE WHEN kpi_name = 'LTIF'                                   THEN value END) AS ltif,
    MAX(CASE WHEN kpi_name = 'LTIF'                         THEN scope_or_note END) AS ltif_note,
    MAX(CASE WHEN kpi_name = 'Revenue'                                THEN value END) AS revenue_eur_m,
    MAX(CAST(is_restated AS INT)) = 1                                                 AS has_restated_values
  FROM workspace.silver.utility_kpis
  GROUP BY company, report_year
),

calculated AS (
  SELECT
    *,
    scope1_t + scope2_market_t                              AS scope12_market_t,
    scope1_t + scope2_market_t + scope3_t                   AS total_ghg_market_t,
    ROUND(female_employees / employees * 100, 1)            AS female_share_pct,
    ROUND((scope1_t + scope2_market_t) / revenue_eur_m, 1)  AS ghg_intensity_t_per_eur_m,
    ROUND(scope1_t / employees, 1)                          AS scope1_t_per_employee,
    ROUND(energy_mwh / revenue_eur_m, 1)                    AS energy_intensity_mwh_per_eur_m,
    ltif_note LIKE '%INCL. contractors%'                    AS ltif_includes_contractors
  FROM pivoted
)

SELECT
  *,
  ROUND(
    (scope12_market_t - LAG(scope12_market_t) OVER (PARTITION BY company ORDER BY report_year))
    / LAG(scope12_market_t) OVER (PARTITION BY company ORDER BY report_year) * 100, 1
  ) AS scope12_change_yoy_pct
FROM calculated;

In [0]:
SELECT company, report_year, female_share_pct, ghg_intensity_t_per_eur_m,
       scope1_t_per_employee, total_ghg_market_t, scope12_change_yoy_pct,
       ltif, ltif_includes_contractors
FROM workspace.gold.company_kpis
ORDER BY company, report_year;